In [ ]:

import torch 
def rope(x, tetha = 10000.0):
        N,H,T,W = x.shape
        ang = torch.arange(0,T,device=x.device).float().view(T,1) * tetha ** ( -2 * torch.arange(0,W//2,device=x.device).float() / W )
        cos = torch.cos(ang)
        sin = torch.sin(ang)
        return torch.stack([
            x[:,:,:,0::2] * cos - x[:,:,:,1::2] * sin,
            x[:,:,:,0::2] * sin + x[:,:,:,1::2] * cos
        ],dim=-1).flatten(-2)

torch.manual_seed = 333
x = torch.randn((1,1,6,4))

print(x)
print(rope(x))

tensor([[[[ 1.2578,  0.4011,  0.3276, -0.6966],
          [ 0.8228,  0.5776, -0.3017, -0.1396],
          [-2.3811, -0.5569,  0.8270, -1.7335],
          [-0.9356,  0.6296, -0.7788,  1.3596],
          [ 1.0453, -0.5230,  0.1849,  2.0359],
          [ 1.2237,  0.4481,  0.5627, -0.4700]]]])
tensor([[[[ 1.2578,  0.3276],
          [-0.0415, -0.3003],
          [ 1.4973,  0.8615],
          [ 0.8374, -0.8193],
          [-1.0790,  0.1033],
          [ 0.7768,  0.5855]]]])
tensor([[[[ 0.4011, -0.6966],
          [ 1.0044, -0.1426],
          [-1.9334, -1.7166],
          [-0.7554,  1.3356],
          [-0.4493,  2.0417],
          [-1.0463, -0.4413]]]])
tensor([[[[ 1.2578,  0.4011,  0.3276, -0.6966],
          [-0.0415,  1.0044, -0.3003, -0.1426],
          [ 1.4973, -1.9334,  0.8615, -1.7166],
          [ 0.8374, -0.7554, -0.8193,  1.3356],
          [-1.0790, -0.4493,  0.1033,  2.0417],
          [ 0.7768, -1.0463,  0.5855, -0.4413]]]])


In [ ]:
from turtle import forward
import torch 
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim 
from tqdm import trange

DEVICE = "mps"

DATA = open("shakespeare.txt", "r").read()

print(len(DATA))

VOCAB = list(sorted(set(DATA)))
VOCAB_I = {v:k for k,v in enumerate(VOCAB)}
VOCAB_SIZE = len(VOCAB)

def encode(s):
    return torch.tensor([VOCAB_I[x] for x in s], device = DEVICE)

def decode(t):
    return ''.join([VOCAB[x] for x in t]) 

DATA_TRAIN = encode(DATA[:int(len(DATA)*0.9)])
DATA_TEST = encode(DATA[int(len(DATA)*0.9):])

def get_batch(data, batch_count, context_width):
    idx = torch.randint(len(data)-context_width, (batch_count, 1), device=data.device)
    d = data[idx+torch.arange(0, context_width+1, device=data.device)]
    return d[:,:-1], d[:,1:]


# Day 3
#  - Bug 1 — causal mask has the wrong shape
#  - Bug 2 — dropout applied to the wrong tensor
#  - Bug 3 — generate runs with grad and in train mode
class MultiHeadAttention(nn.Module):
    def __init__(self, context_window, embedding_dim, headsize, dropout_p):
        super().__init__()
        assert embedding_dim % headsize == 0
        self.headsize = headsize
        self.heads = embedding_dim // headsize 
        self.qkv = nn.Linear(embedding_dim, embedding_dim * 3, bias = False)
        self.proj = nn.Linear(embedding_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout_p)
        self.register_buffer("causal_mask", torch.tril(torch.ones(context_window, context_window, dtype=torch.bool)))
        # Rope
        ang = torch.arange(0,context_window).float().view(context_window,1) * 1000.0 ** ( -2 * torch.arange(0,headsize//2).float() / headsize )
        self.register_buffer("rope_cos", torch.cos(ang), persistent=False)
        self.register_buffer("rope_sin", torch.sin(ang), persistent=False)

    def rope(self, x):
        T = x.shape[-2]
        xx, yy = torch.chunk(x, 2, dim=-1)
        cos = self.rope_cos[:T,:]
        sin = self.rope_sin[:T,:]
        return torch.cat([
            xx * cos - yy * sin,
            xx * sin + yy * cos
        ],dim=-1)

    def forward(self, x):
        B,T,E = x.shape
        H,W = self.heads, self.headsize 
        q,k,v = self.qkv(x).view(B,T,3,H,W).permute(2,0,3,1,4) # B,H,T,W
        q = self.rope(q)
        k = self.rope(k)
        a = F.scaled_dot_product_attention(q, k, v, 
            dropout_p=self.dropout.p if self.training else 0.0, is_causal=True)
        #a = (q @ k.transpose(-2,-1)) / W**0.5
        #a = a.masked_fill(~self.causal_mask[:T,:T], float('-inf'))
        #a = F.softmax(a, dim=-1)
        #a = self.dropout(a)
        #a = a @ v  # B,H,T,W
        return self.proj(a.transpose(-3,-2).reshape(B,T,E))
        

class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.lin = nn.Linear(embedding_dim, embedding_dim * 4)
        self.gelu = nn.GELU()
        self.proj = nn.Linear(embedding_dim * 4, embedding_dim)

    def forward(self, x):
        x = self.lin(x)
        x = self.gelu(x)
        x = self.proj(x)
        return x


class AttentionBlock(nn.Module):
    def __init__(self, context_window, embedding_dim, headsize, dropout_p):
        super().__init__()
        self.mha = MultiHeadAttention(context_window, embedding_dim, headsize, dropout_p)
        self.layn1 = nn.LayerNorm(embedding_dim)
        self.ff = FeedForward(embedding_dim)
        self.layn2 = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, x):
        x = x + self.dropout(self.mha(self.layn1(x)))
        x = x + self.dropout(self.ff(self.layn2(x)))
        return x


class Transformer(nn.Module):
    def __init__(self, vocabulary_size, context_window, embedding_dim, attention_blocks, headsize, dropout_p):
        super().__init__()
        self.context_window = context_window
        self.embeddings = nn.Embedding(vocabulary_size, embedding_dim)
        #self.embeddings_pos = nn.Embedding(context_window, embedding_dim)
        self.ablocks = nn.Sequential(
            *[AttentionBlock(context_window, embedding_dim, headsize, dropout_p) 
            for _ in range(attention_blocks)])
        self.layn = nn.LayerNorm(embedding_dim)
        self.lmh = nn.Linear(embedding_dim, vocabulary_size, bias=False)
        self.dropout = nn.Dropout(dropout_p)
        self.embeddings.weight = self.lmh.weight
        self.register_buffer("arange_pos", torch.arange(0, context_window))

    def forward(self, x, y = None):
        B,T = x.shape
        x = self.embeddings(x)# + self.embeddings_pos(self.arange_pos[:T])
        x = self.dropout(x)
        x = self.ablocks(x)
        x = self.layn(x)
        x = self.lmh(x)
        if y is None:
            return x
        else:
            return F.cross_entropy(x.view(B*T,-1), y.reshape(-1))

    @torch.no_grad()
    def generate(self, limit, prompt="", device=DEVICE):
        was_training = self.training
        self.eval()
        out = [0] if len(prompt) == 0 else encode(prompt)
        for _ in range(limit):
            x = torch.tensor(out[-self.context_window:], device=device).view(1,-1)
            y = self(x)
            y = y[:,-1]
            w = torch.multinomial(F.softmax(y, dim=-1), 1)
            out.append(w.item())
        self.train(was_training)
        return decode(out[1:]) if len(prompt) == 0 else decode(out)

@torch.no_grad()
def estimate_loss(m):
    was_training = m.training
    m.eval()
    res = [
        m(*get_batch(data, 100, m.context_window))
        for data in (DATA_TEST, DATA_TRAIN)
    ]
    m.train(was_training)
    return res


m = Transformer(
        VOCAB_SIZE, 
        context_window=64, 
        embedding_dim=128, 
        headsize=32, 
        attention_blocks=8, 
        dropout_p=0.1
    ).to(DATA_TRAIN.device)

print(estimate_loss(m))

o = optim.AdamW(m.parameters(), 1e-3)

def train_loop(o, m, batch_size, iterations):
    progress = trange(iterations, desc="Training")
    for i in progress:
        o.zero_grad()
        loss = m(*get_batch(DATA_TRAIN, batch_size, m.context_window))
        loss.backward()
        #torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        o.step()
        if i%100==99:
            l_ts,l_tr = estimate_loss(m)
            progress.set_postfix({
                "loss_test": f"{l_ts:.4f}",
                "loss_train": f"{l_tr:.4f}"
            })

train_loop(o,m,256,1000)

# 1.6693
# 1.5808 @ 01:34
# 1.5327 @ 01:03
# 1.4970 @ 00:57

print(estimate_loss(m))

m.eval()
print(m.generate(1000))
m.train()
print()

1115394
[tensor(4.2117, device='mps:0'), tensor(4.2316, device='mps:0')]


Training: 100%|██████████| 1000/1000 [00:56<00:00, 17.61it/s, loss_test=1.6426, loss_train=1.4771]


[tensor(1.6715, device='mps:0'), tensor(1.4530, device='mps:0')]
First Sestable:
My lords, to be onfilly. Which a sunjust not of them,
He devilress and true, the king.

POMSTER:
From hide, in shis so's your bloody;
The posson with duty of to the holdsing amest vow
So paint; make her, lord of hear day herself
Grenous messNow hild ful in with
Non for this very sons it in his ma3
Nursure he struce, what you may breakry!

DUCHESS OF ARD III:
I had my milver, troom there, by it bound
Of noin-made of did being thing look ress in
thy orous to the withers answay, how contently
Are not ins'erly be, sure.

LADY GREY:
Come, to make her haste he kin son: tjust;
The sland as thou contenpses to, old leave them to proud
As he appreed cursinal spitge.

First Lord:
And make what thou wrath thee to FARILR:
A stand BAnish's lame, and pressant,
Nay, for a fapt upon the Boli: we will not brother.

WARWICK:
Here with them Clarence, I canus that his corish.
Why, is the streesy, to I safe reserve.

SICINIUS:


In [105]:
A = torch.tril(torch.ones(4,4))
B = torch.randn((1,2,4,4))

B=B.masked_fill(A==0.0, float('-inf'))

print(A)
print(B)

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])
tensor([[[[ 0.9646,    -inf,    -inf,    -inf],
          [-0.0074, -0.7588,    -inf,    -inf],
          [-0.0547, -0.4877, -0.7597,    -inf],
          [-1.0998, -0.7024, -1.5837, -0.1522]],

         [[-0.9107,    -inf,    -inf,    -inf],
          [-0.0808, -0.2134,    -inf,    -inf],
          [ 0.8593,  0.1455,  0.8200,    -inf],
          [ 0.4056, -1.2075, -0.1183, -0.2768]]]])
